In [ ]:
!git clone https://github.com/feralvam/easse.git
%cd easse
!pip install -e .


Cloning into 'easse'...
remote: Enumerating objects: 1964, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 1964 (delta 118), reused 104 (delta 104), pack-reused 1819 (from 1)
Receiving objects: 100% (1964/1964), 33.15 MiB | 17.37 MiB/s, done.
Resolving deltas: 100% (1231/1231), done.
/content/easse
Obtaining file:///content/easse
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/facebookresearch/text-simplification-evaluation.git (to revision main) to /tmp/pip-install-9w8nk9go/tseval_df362c2f4bd74725b2971db7cf2a5030
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/text-simplification-evaluation.git /tmp/pip-install-9w8nk9go/tseval_df362c2f4bd74725b2971db7cf2a5030
  Resolved https://github.com/facebookresearch/text-simplification-evaluation.git to commit dea8863683ea5946fd50184883c9be7a7339e821
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━

In [ ]:
import pandas as pd

data = pd.read_parquet("hf://datasets/turk/simplification/")

df_test = data[2000:].reset_index(drop=True)

print("Total de muestras:", len(df_test))
print("Ejemplo:")
print("Original:", df_test.iloc[0]["original"])
print("Simplificaciones:", df_test.iloc[0]["simplifications"])


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Total de muestras: 359
Ejemplo:
Original: the contemporary status of the mandaeans has prompted a number of american intellectuals and civil rights activists to call for their government to extend refugee status to the community .
Simplificaciones: ['a number of american civil rights activists and intellectuals have called for their government to extend refugee status to the mandaeans .'
 'the current status of the mandaeans has caused a number of american intellectuals and civil rights activists to call for their government to extend refugee status to the community .'
 'the standing , today , of those that practice the ancient religion , known as mandaeism , has led a lot of american intellectuals and civil rights activists to call to their governmen to offer refugee status to the community .'
 'the status of the mandaenas has prompted a number of american intellectuals and civil rights activists to call for their government to extend refugee status to the community .'
 'the modern st

In [ ]:
def expand_data(df):
    new_rows = []
    df = df.rename(columns={'simplifications': 'simplification'})
    for _, row in df.iterrows():
        for s in row['simplification']:
            new_rows.append({'original': row['original'], 'simplification': s})
    return pd.DataFrame(new_rows)

df_train = expand_data(df_test)


In [ ]:
from datasets import Dataset
from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("t5-base")

hf_dataset = Dataset.from_pandas(df_train[["original", "simplification"]])

def preprocess(example):
    input_text = "simplify: " + example["original"]
    target_text = example["simplification"]

    model_inputs = tokenizer(input_text, max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(target_text, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = hf_dataset.map(preprocess, batched=False)


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/2872 [00:00<?, ? examples/s]

In [ ]:
from transformers import T5ForConditionalGeneration, TrainingArguments, Trainer

model = T5ForConditionalGeneration.from_pretrained("t5-base")

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="no",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"


In [ ]:
trainer.train()


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
10,8.852800
20,3.579700
30,0.875400
40,0.529600
50,0.337000
60,0.329600
70,0.292600
80,0.278300
90,0.221500
100,0.227800


TrainOutput(global_step=1077, training_loss=0.2961640470871354, metrics={'train_runtime': 609.0338, 'train_samples_per_second': 14.147, 'train_steps_per_second': 1.768, 'total_flos': 1311695296266240.0, 'train_loss': 0.2961640470871354, 'epoch': 3.0})

In [ ]:
model.save_pretrained("./t5-finetuned-simplification")
tokenizer.save_pretrained("./t5-finetuned-simplification")


('./t5-finetuned-simplification/tokenizer_config.json',
 './t5-finetuned-simplification/special_tokens_map.json',
 './t5-finetuned-simplification/spiece.model',
 './t5-finetuned-simplification/added_tokens.json')

In [ ]:
inputs = ["simplify: " + s for s in df_test["original"]]
tokenized_inputs = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True, max_length=128)
tokenized_inputs = {k: v.to(model.device) for k, v in tokenized_inputs.items()}

outputs = model.generate(**tokenized_inputs, max_length=128)
df_test["generated"] = tokenizer.batch_decode(outputs, skip_special_tokens=True)


In [ ]:
from easse.sari import corpus_sari
from easse.bleu import sentence_bleu
from easse.fkgl import corpus_fkgl
import numpy as np
from tqdm.auto import tqdm
import nltk
nltk.download('punkt_tab')

df_eval = df_test.dropna(subset=["generated"]).reset_index(drop=True)

saris = []
bleus = []

for i in tqdm(range(len(df_eval))):
    sari_score = corpus_sari(
        orig_sents=[df_eval.loc[i, "original"]],
        sys_sents=[df_eval.loc[i, "generated"]],
        refs_sents=[[ref] for ref in df_eval.loc[i, "simplifications"]],
        use_paper_version=True
    )
    saris.append(sari_score)

    bleu_score = sentence_bleu(
        sys_sent=df_eval.loc[i, "generated"],
        ref_sents=df_eval.loc[i, "simplifications"]
    )
    bleus.append(bleu_score)

fkgl_score = corpus_fkgl(df_eval["generated"].tolist())

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


  0%|          | 0/359 [00:00<?, ?it/s]

In [ ]:
print(f"SARI (↑): {np.mean(saris):.3f}")
print(f"BLEU (↑): {np.mean(bleus):.3f}")
print(f"FKGL (↓): {fkgl_score:.3f}")


SARI (↑): 30.974
BLEU (↑): 97.044
FKGL (↓): 9.533
